# Sesión 4 · Datos tabulares II: unir tablas y ordenar el código

**Programación avanzada en Python aplicada al análisis social** · viernes 11 de diciembre de 2026

Objetivos:

1. Resumir una tabla por grupos con `groupby`.
2. **Unir** dos tablas con `merge` y comprobar que la unión ha salido bien.
3. Guardar vuestras funciones en un **fichero propio** e importarlas desde cualquier notebook.

In [ ]:
# Preparacion: localiza la carpeta de la asignatura (y la descarga si estas en Google Colab).
# Ejecuta esta celda primero; no hace falta entenderla todavia.
import os, subprocess
from pathlib import Path

REPO = "https://github.com/luisaugustos/ppy-analisis-social.git"


def _buscar_carpeta():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        for candidata in (carpeta, carpeta / "ppy-analisis-social"):
            if (candidata / "datos" / "ORIGEN.md").is_file():
                return candidata
    return None


raiz = _buscar_carpeta()
if raiz is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    raiz = Path("ppy-analisis-social").resolve()

os.chdir(raiz)
print("Carpeta de trabajo lista. Los datos estan en la subcarpeta 'datos'.")

In [ ]:
import pandas as pd

hipotecas = pd.read_csv("datos/hipotecas.csv", dtype={"cod_prov": str}, parse_dates=["fecha"])
hipotecas = hipotecas[hipotecas["cod_prov"] != "00"]          # sin el total nacional
hipotecas["anyo"] = hipotecas["fecha"].dt.year
hipotecas.head(3)

## 1. Resumir por grupos: `groupby`

`groupby` hace tres cosas: **divide** la tabla en grupos, **aplica** un cálculo a cada uno y **junta** los resultados.

In [ ]:
hipotecas.groupby("anyo")["hipotecas"].sum().tail(4)

Se lee de izquierda a derecha: «agrupa por año, toma la columna hipotecas y súmala».

Se puede agrupar por varias columnas y pedir varios cálculos a la vez, dando nombre a cada resultado:

In [ ]:
anual = (
    hipotecas
    .groupby(["cod_prov", "provincia", "ccaa", "anyo"], as_index=False)
    .agg(
        hipotecas=("hipotecas", "sum"),
        importe_miles=("importe_miles", "sum"),
        meses=("fecha", "count"),
    )
)
anual.tail(3)

> **Encadenar con paréntesis**
>
>  Al poner toda la expresión entre paréntesis, cada paso puede ir en su línea. Se lee como una receta: «toma hipotecas, agrúpala, calcula».

La columna `meses` delata que el último año está incompleto. Nos quedamos con los años de doce meses:

In [ ]:
anual = anual[anual["meses"] == 12]
print(anual["anyo"].min(), "-", anual["anyo"].max())

## 2. Unir tablas: `merge`

Para comparar provincias de tamaño distinto necesitamos la población, que está en otro fichero.

In [ ]:
poblacion = pd.read_csv("datos/poblacion.csv", dtype={"cod_prov": str})
poblacion.head(3)

Las dos tablas comparten las columnas `cod_prov` y `anyo`: son la **clave** de la unión. Antes de unir conviene hacerse tres preguntas:

1. ¿La clave tiene el **mismo tipo y formato** en las dos tablas? (por eso leemos `cod_prov` como texto en ambas)
2. ¿Cuántas filas espero? Aquí, una de población por cada fila de hipotecas.
3. ¿Qué hago con las filas que **no encuentren pareja**?

In [ ]:
union = anual.merge(
    poblacion[["cod_prov", "anyo", "poblacion"]],
    on=["cod_prov", "anyo"],        # la clave
    how="left",                     # conservo todas las filas de la tabla de la izquierda
    validate="one_to_one",          # error si la clave está repetida
    indicator=True,                 # añade una columna que dice si hubo pareja
)
union["_merge"].value_counts()

`indicator=True` añade la columna `_merge`. Hay filas `left_only`: hipotecas sin dato de población. ¿Cuáles?

In [ ]:
sorted(union.loc[union["_merge"] == "left_only", "anyo"].unique())

La tabla de población termina antes que la de hipotecas. No es un error del código, es una **decisión** que hay que tomar y dejar escrita. Aquí usaremos la última población conocida de cada provincia.

In [ ]:
union = union.drop(columns="_merge").sort_values(["cod_prov", "anyo"])
union["poblacion"] = union.groupby("cod_prov")["poblacion"].ffill()      # arrastra el último valor conocido
union["tasa_mil"] = union["hipotecas"] / union["poblacion"] * 1000

union[union["anyo"] == 2025].nlargest(5, "tasa_mil")[["provincia", "hipotecas", "poblacion", "tasa_mil"]].round(2)

**Tabla. Qué filas conserva cada tipo de unión.**

| `how` | Conserva | Cuándo usarlo |
| --- | --- | --- |
| `"left"` | Todas las filas de la tabla izquierda | Añadir información a la tabla principal |
| `"inner"` | Solo las filas con pareja en ambas | Cuando se necesitan todas las variables |
| `"outer"` | Todas las filas de las dos tablas | Para investigar qué claves no coinciden |

> **Unid por código, no por nombre**
>
>  «Rioja, La» y «La Rioja», «Araba/Álava» y «Álava»: los nombres cambian de una fuente a otra. Siempre que exista un código oficial, usadlo como clave.

## 3. De largo a ancho

La tabla `union` está en formato **largo**: una fila por provincia y año. Para leerla o compararla viene mejor el formato **ancho**, con un año en cada columna.

In [ ]:
ancho = union.pivot(index="provincia", columns="anyo", values="tasa_mil").round(2)
ancho.loc[["Granada", "Salamanca"], [2019, 2022, 2025]]

## 4. Ordenar el código: vuestro propio módulo

Ya tenéis varias funciones repartidas por los notebooks de estas semanas. Copiarlas y pegarlas de uno a otro acaba mal: se arregla un fallo en una copia y se olvida en las demás.

La solución es guardarlas en un fichero `.py`, un **módulo**, e importarlo igual que importamos pandas.

En un notebook, `%%writefile` guarda el contenido de la celda en un fichero:

In [ ]:
%%writefile vivienda.py
"""Funciones para trabajar con los datos de vivienda del curso."""
import pandas as pd


def leer_hipotecas(ruta="datos/hipotecas.csv"):
    """Lee las hipotecas mensuales por provincia, sin el total nacional."""
    tabla = pd.read_csv(ruta, dtype={"cod_prov": str}, parse_dates=["fecha"])
    tabla = tabla[tabla["cod_prov"] != "00"]
    tabla["anyo"] = tabla["fecha"].dt.year
    return tabla


def resumen_anual(hipotecas):
    """Suma las hipotecas de cada provincia por año y se queda con los años completos."""
    anual = (
        hipotecas
        .groupby(["cod_prov", "provincia", "ccaa", "anyo"], as_index=False)
        .agg(hipotecas=("hipotecas", "sum"), importe_miles=("importe_miles", "sum"), meses=("fecha", "count"))
    )
    return anual[anual["meses"] == 12].drop(columns="meses")


def anadir_tasa(anual, ruta_poblacion="datos/poblacion.csv"):
    """Añade la población y las hipotecas por mil habitantes.

    En los años sin dato de población se usa el último conocido de cada provincia.
    """
    poblacion = pd.read_csv(ruta_poblacion, dtype={"cod_prov": str})
    union = anual.merge(poblacion[["cod_prov", "anyo", "poblacion"]], on=["cod_prov", "anyo"],
                        how="left", validate="one_to_one")
    union = union.sort_values(["cod_prov", "anyo"])
    union["poblacion"] = union.groupby("cod_prov")["poblacion"].ffill()
    union["tasa_mil"] = union["hipotecas"] / union["poblacion"] * 1000
    return union

El fichero `vivienda.py` ya existe en la carpeta de trabajo. Ahora se puede importar:

In [ ]:
import vivienda

tabla = vivienda.anadir_tasa(vivienda.resumen_anual(vivienda.leer_hipotecas()))
tabla[tabla["provincia"] == "Salamanca"].tail(3).round(2)

Todo el trabajo de la sesión cabe ahora en **una línea** que se lee sola. Y `help` muestra vuestra propia documentación:

In [ ]:
help(vivienda.anadir_tasa)

> **Si cambiáis el fichero**
>
>  Python solo lee un módulo la primera vez que se importa. Si editáis `vivienda.py`, reiniciad el notebook o ejecutad:
>
>     import importlib
>     importlib.reload(vivienda)

### Cómo queda el proyecto

```
mi_proyecto/
├── datos/              <- ficheros originales, sin tocar
│   └── descargas/      <- copias de lo que descarga el código
├── resultados/         <- tablas y gráficos generados
├── vivienda.py         <- vuestras funciones
└── analisis.ipynb      <- el notebook: importa, usa y comenta
```

El notebook cuenta la historia; el módulo guarda las herramientas.

## 5. Guardar el resultado

Para una tabla intermedia, el formato **Parquet** es mejor que CSV: ocupa menos y, sobre todo, **recuerda los tipos** (las fechas siguen siendo fechas y los códigos, texto).

In [ ]:
from pathlib import Path

Path("resultados").mkdir(exist_ok=True)
tabla.to_parquet("resultados/vivienda_anual.parquet", index=False)

releida = pd.read_parquet("resultados/vivienda_anual.parquet")
print(releida["cod_prov"].head(2).tolist())           # siguen siendo '01', '01': no se han vuelto números

## Resumen

- `groupby(...).agg(...)` resume por grupos; revisad siempre cuántas observaciones tiene cada grupo.
- En cada `merge`: clave limpia, `how` explícito, `validate` e `indicator` hasta estar seguros.
- Las funciones que usáis más de una vez van a un **módulo** `.py`; el notebook lo importa.
- Tablas intermedias en **Parquet**.

Los *métodos* para tratar valores perdidos o reducir datos se estudian en *Obtención, almacenamiento y preprocesamiento*; aquí nos ocupamos de que el código que los aplica sea claro y reutilizable.